In [1]:
%pip install seaborn lightgbm

Note: you may need to restart the kernel to use updated packages.


# 🚢 Titanic ML Pipeline - 모델링 및 Artifacts 생성

이 노트북은 Titanic 데이터를 학습하고, 결과 artifacts를 정해진 규칙에 따라 로컬 output 폴더에 생성합니다.

## Output 구조
```
output/{run_id}/
    ├── metadata/run_manifest.yml
    ├── config/env.yml, meta.yml, model.yml
    ├── data_refs/input_data_ref.yml
    ├── artifacts/
    │   ├── model/model_baseline.pkl, model_baseline.txt
    │   ├── metrics/model_metrics.yml
    │   ├── charts/*.png
    │   └── explainability/shap_summary.png
    └── reports/classification_report.html
```

## 1. 환경 설정 및 Import

In [2]:
import os
import yaml
import json
import hashlib
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from uuid import uuid4

# ML 라이브러리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, roc_curve,
    classification_report
)
import lightgbm as lgb

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ 라이브러리 임포트 완료")

✅ 라이브러리 임포트 완료


## 2. 경로 및 설정 로드

In [3]:
# 디렉토리 설정
BASE_DIR = Path.cwd()
CONF_DIR = BASE_DIR / 'conf'
DATA_DIR = BASE_DIR / 'data'

def load_yaml(filepath: Path) -> dict:
    with open(filepath, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def save_yaml(data: dict, filepath: Path):
    with open(filepath, 'w', encoding='utf-8') as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

# 설정 파일 로드
env_config = load_yaml(CONF_DIR / 'env.yml')
meta_config = load_yaml(CONF_DIR / 'meta.yml')
model_config = load_yaml(CONF_DIR / 'model.yml')

print("✅ 설정 파일 로드 완료")
print(f"   Project: {meta_config['project']}")
print(f"   User Id: {meta_config['user_id']}")
print(f"   Experiment: {meta_config['experiment']}")
print(f"   Algorithm: {model_config['algorithm']['name']}")

✅ 설정 파일 로드 완료
   Project: titanic-survival-prediction
   User Id: sample
   Experiment: baseline-awesome-sean-v1
   Algorithm: lightgbm


In [4]:
# 이 셀에 반드시 "parameters" 태그 달아야 함 -> 노트북 커널 실행시에는 별도로 건드리지 않아도됨
run_id = None        # run_pm.py에서 주입됨
output_dir = None    # run_pm.py에서 주입됨

In [5]:
# Parameters
run_id = "20260320_lightgbm_baseline_a9aa9fd3"
output_dir = "/home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3"


In [6]:
# Run ID 생성
RUN_DATE = datetime.now().strftime('%Y%m%d')
RUN_UUID = uuid4().hex[:8]
ALGORITHM = model_config['algorithm']['name']
SUFFIX = model_config['algorithm']['suffix']
# 주입된 run_id 있으면 사용, 없으면 자체 생성
if run_id is None:
    RUN_ID = f"{RUN_DATE}_{ALGORITHM}_{SUFFIX}_{RUN_UUID}"
else:
    RUN_ID = run_id

# 타임스탬프
CREATED_AT = datetime.now().isoformat()

print(f"🏷️  Run ID: {RUN_ID}")
print(f"⏰ Created: {CREATED_AT}")

🏷️  Run ID: 20260320_lightgbm_baseline_a9aa9fd3
⏰ Created: 2026-03-20T01:33:24.985458


In [7]:
# Output 디렉토리 생성
if output_dir is None:
    OUTPUT_ROOT = BASE_DIR / env_config["local"]["output_dir"] / RUN_ID
else:
    OUTPUT_ROOT = Path(output_dir)

OUTPUT_DIRS = {
    'metadata': OUTPUT_ROOT / 'metadata',
    'config': OUTPUT_ROOT / 'config',
    'data_refs': OUTPUT_ROOT / 'data_refs',
    'model': OUTPUT_ROOT / 'artifacts' / 'model',
    'metrics': OUTPUT_ROOT / 'artifacts' / 'metrics',
    'charts': OUTPUT_ROOT / 'artifacts' / 'charts',
    'explainability': OUTPUT_ROOT / 'artifacts' / 'explainability',
    'reports': OUTPUT_ROOT / 'reports'
}

for name, path in OUTPUT_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"   📁 {name}: {path}")

print(f"\n✅ Output 디렉토리 생성 완료: {OUTPUT_ROOT}")

   📁 metadata: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/metadata
   📁 config: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/config
   📁 data_refs: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/data_refs
   📁 model: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/artifacts/model
   📁 metrics: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/artifacts/metrics
   📁 charts: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/artifacts/charts
   📁 explainability: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/artifacts/explainability
   📁 reports: /home/ec2-user/SageMaker

## 3. 데이터 로드 및 전처리

In [8]:
# 데이터 로드
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

# validation.csv가 있으면 사용, 없으면 train에서 분할
validation_path = DATA_DIR / 'validation.csv'
if validation_path.exists():
    valid_df = pd.read_csv(validation_path)
    print(f"✅ validation.csv 로드 완료")
else:
    valid_df = None
    print(f"ℹ️  validation.csv 없음 - train에서 분할 예정")

print(f"\n📊 데이터 크기")
print(f"   Train: {len(train_df)} rows")
print(f"   Test:  {len(test_df)} rows")
if valid_df is not None:
    print(f"   Valid: {len(valid_df)} rows")

✅ validation.csv 로드 완료

📊 데이터 크기
   Train: 891 rows
   Test:  418 rows
   Valid: 418 rows


In [9]:
def preprocess_data(df, is_train=True):
    """데이터 전처리"""
    df = df.copy()
    
    # 피처 설정
    features_config = model_config['features']
    target_col = features_config['target_col']
    numeric_cols = features_config['numeric_col']
    categorical_cols = features_config['categorical_col']
    drop_cols = features_config['drop_col']
    
    # 결측치 처리
    missing_config = model_config['preprocessing']['missing_values']
    if 'Age' in df.columns:
        df['Age'] = df['Age'].fillna(df['Age'].median())
    if 'Embarked' in df.columns:
        df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    if 'Fare' in df.columns:
        df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    
    # 파생 피처
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']
    
    # Sex 인코딩
    df['Sex'] = df['Sex'].map({'male': 1, 'female': 0})
    
    # Embarked 원핫 인코딩
    embarked_dummies = pd.get_dummies(df['Embarked'], prefix='Embarked')
    df = pd.concat([df, embarked_dummies], axis=1)
    
    # 사용할 피처 선택
    feature_cols = numeric_cols + ['Sex', 'Pclass', 'FamilySize', 'IsAlone', 'FarePerPerson']
    feature_cols += [col for col in df.columns if col.startswith('Embarked_')]
    
    X = df[feature_cols]
    y = df[target_col] if target_col in df.columns else None
    
    return X, y, feature_cols

print("✅ 전처리 함수 정의 완료")

✅ 전처리 함수 정의 완료


In [10]:
# 전처리 수행
X_train_full, y_train_full, feature_cols = preprocess_data(train_df, is_train=True)

# Train/Valid 분할
SEED = model_config['data']['seed']
VALID_RATIO = model_config['data']['train_valid_split']

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full,
    test_size=VALID_RATIO,
    random_state=SEED,
    stratify=y_train_full
)

print(f"✅ 데이터 분할 완료")
print(f"   Train: {len(X_train)} samples")
print(f"   Valid: {len(X_valid)} samples")
print(f"   Features: {len(feature_cols)}")
print(f"\n   Feature list: {feature_cols}")

✅ 데이터 분할 완료
   Train: 712 samples
   Valid: 179 samples
   Features: 12

   Feature list: ['Age', 'Fare', 'SibSp', 'Parch', 'Sex', 'Pclass', 'FamilySize', 'IsAlone', 'FarePerPerson', 'Embarked_C', 'Embarked_Q', 'Embarked_S']


## 4. 모델 학습

In [11]:
# LightGBM 데이터셋 생성
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

# 하이퍼파라미터
params = {
    'objective': model_config['hyperparameters']['objective'],
    'boosting': model_config['hyperparameters']['boosting'],
    'max_depth': model_config['hyperparameters']['max_depth'],
    'learning_rate': model_config['hyperparameters']['learning_rate'],
    'num_leaves': model_config['hyperparameters']['num_leaves'],
    'feature_fraction': model_config['hyperparameters']['feature_fraction'],
    'bagging_fraction': model_config['hyperparameters']['bagging_fraction'],
    'bagging_freq': model_config['hyperparameters']['bagging_freq'],
    'verbosity': -1,
    'seed': SEED,
    'metric': ['auc', 'binary_logloss']
}

print("📊 하이퍼파라미터:")
for k, v in params.items():
    print(f"   {k}: {v}")

📊 하이퍼파라미터:
   objective: binary
   boosting: gbdt
   max_depth: 6
   learning_rate: 0.05
   num_leaves: 31
   feature_fraction: 0.8
   bagging_fraction: 0.8
   bagging_freq: 5
   verbosity: -1
   seed: 2026
   metric: ['auc', 'binary_logloss']


In [12]:
# 학습 기록 저장용
evals_result = {}

# 모델 학습
print("\n🚀 모델 학습 시작...")
model = lgb.train(
    params,
    train_data,
    num_boost_round=model_config['hyperparameters']['num_iterations'],
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.log_evaluation(period=50),
        lgb.record_evaluation(evals_result)
    ]
)

COMPLETED_AT = datetime.now().isoformat()
print(f"\n✅ 학습 완료!")
print(f"   Best iteration: {model.best_iteration}")


🚀 모델 학습 시작...


[50]	train's auc: 0.930578	train's binary_logloss: 0.343151	valid's auc: 0.85639	valid's binary_logloss: 0.427387


[100]	train's auc: 0.951592	train's binary_logloss: 0.282467	valid's auc: 0.866864	valid's binary_logloss: 0.426673
[150]	train's auc: 0.966507	train's binary_logloss: 0.245307	valid's auc: 0.860804	valid's binary_logloss: 0.44404


[200]	train's auc: 0.974668	train's binary_logloss: 0.217155	valid's auc: 0.865152	valid's binary_logloss: 0.452804



✅ 학습 완료!
   Best iteration: 0


## 5. 모델 평가

In [13]:
def evaluate_model(model, X, y, dataset_name=''):
    """모델 평가 메트릭 계산"""
    y_pred_proba = model.predict(X)
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    metrics = {
        'accuracy': round(accuracy_score(y, y_pred), 4),
        'precision': round(precision_score(y, y_pred), 4),
        'recall': round(recall_score(y, y_pred), 4),
        'f1_score': round(f1_score(y, y_pred), 4),
        'auc_roc': round(roc_auc_score(y, y_pred_proba), 4),
        'log_loss': round(log_loss(y, y_pred_proba), 4),
        'samples': len(y)
    }
    
    print(f"\n📊 {dataset_name} Metrics:")
    for k, v in metrics.items():
        print(f"   {k}: {v}")
    
    return metrics, y_pred, y_pred_proba

# 평가 수행
train_metrics, y_train_pred, y_train_proba = evaluate_model(model, X_train, y_train, 'Train')
valid_metrics, y_valid_pred, y_valid_proba = evaluate_model(model, X_valid, y_valid, 'Valid')


📊 Train Metrics:
   accuracy: 0.9199
   precision: 0.9426
   recall: 0.8425
   f1_score: 0.8897
   auc_roc: 0.9747
   log_loss: 0.2172
   samples: 712



📊 Valid Metrics:
   accuracy: 0.8324
   precision: 0.8305
   recall: 0.7101
   f1_score: 0.7656
   auc_roc: 0.8652
   log_loss: 0.4528
   samples: 179


In [14]:
# 혼동 행렬
cm = confusion_matrix(y_valid, y_valid_pred)
tn, fp, fn, tp = cm.ravel()

print(f"\n📊 Confusion Matrix (Valid):")
print(f"   TN: {tn}, FP: {fp}")
print(f"   FN: {fn}, TP: {tp}")
print(f"   Specificity: {tn / (tn + fp):.4f}")
print(f"   Sensitivity: {tp / (tp + fn):.4f}")


📊 Confusion Matrix (Valid):
   TN: 100, FP: 10
   FN: 20, TP: 49
   Specificity: 0.9091
   Sensitivity: 0.7101


In [15]:
# 피처 중요도
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

feature_importance['importance'] = feature_importance['importance'] / feature_importance['importance'].sum()
feature_importance['rank'] = range(1, len(feature_importance) + 1)

print("\n📊 Feature Importance (Top 10):")
print(feature_importance.head(10).to_string(index=False))


📊 Feature Importance (Top 10):
      feature  importance  rank
          Sex    0.299821     1
          Age    0.187071     2
         Fare    0.181146     3
FarePerPerson    0.166054     4
       Pclass    0.080783     5
   FamilySize    0.037067     6
   Embarked_S    0.016134     7
        SibSp    0.011351     8
        Parch    0.008275     9
   Embarked_Q    0.005316    10


---

## 6. Artifacts 생성

### 6.1 Config 복사

In [16]:
import shutil

# 설정 파일 복사
for config_file in ['env.yml', 'meta.yml', 'model.yml']:
    src = CONF_DIR / config_file
    dst = OUTPUT_DIRS['config'] / config_file
    shutil.copy(src, dst)
    print(f"   ✅ {config_file} -> {dst}")

print("\n✅ Config 파일 복사 완료")

   ✅ env.yml -> /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/config/env.yml
   ✅ meta.yml -> /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/config/meta.yml
   ✅ model.yml -> /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/config/model.yml

✅ Config 파일 복사 완료


### 6.2 Model 저장

In [17]:
# Pickle 형식 저장
model_pkl_path = OUTPUT_DIRS['model'] / f'model_{SUFFIX}.pkl'
with open(model_pkl_path, 'wb') as f:
    pickle.dump(model, f)
print(f"   ✅ {model_pkl_path}")

# LightGBM 텍스트 형식 저장
model_txt_path = OUTPUT_DIRS['model'] / f'model_{SUFFIX}.txt'
model.save_model(str(model_txt_path))
print(f"   ✅ {model_txt_path}")

# 모델 크기 확인
model_size_mb = model_pkl_path.stat().st_size / (1024 * 1024)
print(f"\n✅ Model 저장 완료 (Size: {model_size_mb:.2f} MB)")

   ✅ /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/artifacts/model/model_baseline.pkl
   ✅ /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3/artifacts/model/model_baseline.txt

✅ Model 저장 완료 (Size: 0.34 MB)


### 6.3 Charts 생성

In [18]:
# 1. Feature Importance Chart
fig, ax = plt.subplots(figsize=(10, 8))
top_features = feature_importance.head(10)
sns.barplot(data=top_features, x='importance', y='feature', palette='Blues_d', ax=ax)
ax.set_title('Feature Importance (Top 10)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.savefig(OUTPUT_DIRS['charts'] / 'feature_importance.png', dpi=150)
plt.close()
print("   ✅ feature_importance.png")

   ✅ feature_importance.png


In [19]:
# 2. ROC Curve
fpr, tpr, thresholds = roc_curve(y_valid, y_valid_proba)
auc_score = roc_auc_score(y_valid, y_valid_proba)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {auc_score:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.fill_between(fpr, tpr, alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIRS['charts'] / 'roc_curve.png', dpi=150)
plt.close()
print("   ✅ roc_curve.png")

   ✅ roc_curve.png


In [20]:
# 3. Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Died (0)', 'Survived (1)'],
            yticklabels=['Died (0)', 'Survived (1)'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix (Validation Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIRS['charts'] / 'confusion_matrix.png', dpi=150)
plt.close()
print("   ✅ confusion_matrix.png")

   ✅ confusion_matrix.png


In [21]:
# 4. Learning Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC
axes[0].plot(evals_result['train']['auc'], label='Train', linewidth=2)
axes[0].plot(evals_result['valid']['auc'], label='Valid', linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('AUC')
axes[0].set_title('Learning Curve - AUC', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Log Loss
axes[1].plot(evals_result['train']['binary_logloss'], label='Train', linewidth=2)
axes[1].plot(evals_result['valid']['binary_logloss'], label='Valid', linewidth=2)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Log Loss')
axes[1].set_title('Learning Curve - Log Loss', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIRS['charts'] / 'learning_curve.png', dpi=150)
plt.close()
print("   ✅ learning_curve.png")

print("\n✅ Charts 생성 완료")

   ✅ learning_curve.png

✅ Charts 생성 완료


### 6.4 Explainability (SHAP-like visualization)

In [22]:
# SHAP 대신 간단한 피처 영향도 시각화
fig, ax = plt.subplots(figsize=(10, 8))

# 피처별 평균값과 중요도 결합
top_10 = feature_importance.head(10)
colors = ['#ff7f0e' if i % 2 == 0 else '#1f77b4' for i in range(len(top_10))]

ax.barh(range(len(top_10)), top_10['importance'].values, color=colors)
ax.set_yticks(range(len(top_10)))
ax.set_yticklabels(top_10['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Feature Impact Summary', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUTPUT_DIRS['explainability'] / 'feature_impact_summary.png', dpi=150)
plt.close()
print("   ✅ feature_impact_summary.png")

print("\n✅ Explainability 생성 완료")

   ✅ feature_impact_summary.png

✅ Explainability 생성 완료


### 6.5 Metrics YAML 생성

In [23]:
# model_metrics.yml 생성
model_metrics = {
    'run_id': RUN_ID,
    'model_name': meta_config['model'],
    'algorithm': ALGORITHM,
    'task': 'binary_classification',
    'evaluated_at': COMPLETED_AT,
    
    'train_metrics': train_metrics,
    'valid_metrics': valid_metrics,
    
    'confusion_matrix': {
        'true_negative': int(tn),
        'false_positive': int(fp),
        'false_negative': int(fn),
        'true_positive': int(tp),
        'specificity': round(tn / (tn + fp), 4),
        'sensitivity': round(tp / (tp + fn), 4)
    },
    
    'feature_importance': [
        {'feature': row['feature'], 'importance': round(row['importance'], 4), 'rank': int(row['rank'])}
        for _, row in feature_importance.head(10).iterrows()
    ],
    
    'training_history': {
        'best_iteration': model.best_iteration,
        'total_iterations': model_config['hyperparameters']['num_iterations'],
        'final_train_auc': round(evals_result['train']['auc'][-1], 4),
        'final_valid_auc': round(evals_result['valid']['auc'][-1], 4)
    },
    
    'model_complexity': {
        'num_trees': model.num_trees(),
        'max_depth': params['max_depth'],
        'num_leaves': params['num_leaves'],
        'total_features': len(feature_cols),
        'model_size_mb': round(model_size_mb, 2)
    }
}

save_yaml(model_metrics, OUTPUT_DIRS['metrics'] / 'model_metrics.yml')
print("   ✅ model_metrics.yml")
print("\n✅ Metrics 저장 완료")

   ✅ model_metrics.yml

✅ Metrics 저장 완료


### 6.6 Data Refs YAML 생성

In [24]:
def get_file_hash(filepath: Path) -> str:
    """파일 SHA256 해시 계산"""
    sha256 = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            sha256.update(chunk)
    return f"sha256:{sha256.hexdigest()[:16]}..."

# input_data_ref.yml 생성
input_data_ref = {
    'source': {
        'type': 'local',
        'path': str(DATA_DIR),
        'version': meta_config['version']
    },
    
    'files': {
        'data': [
            {
                'path': 'data/train.csv',
                'size_bytes': (DATA_DIR / 'train.csv').stat().st_size,
                'row_count': len(train_df),
                'checksum': get_file_hash(DATA_DIR / 'train.csv')
            }
        ]
    },
    
    'schema': {
        'index_col': model_config['features']['index_col'],
        'target_col': model_config['features']['target_col'],
        'feature_count': len(feature_cols)
    },
    
    'split': {
        'method': 'stratified_random',
        'train_ratio': 1 - VALID_RATIO,
        'valid_ratio': VALID_RATIO,
        'seed': SEED,
        'train': {
            'samples': len(X_train),
            'target_distribution': dict(y_train.value_counts().sort_index())
        },
        'valid': {
            'samples': len(X_valid),
            'target_distribution': dict(y_valid.value_counts().sort_index())
        }
    },
    
    'quality': {
        'total_rows': len(train_df),
        'total_columns': len(train_df.columns),
        'used_features': len(feature_cols)
    }
}

save_yaml(input_data_ref, OUTPUT_DIRS['data_refs'] / 'input_data_ref.yml')
print("   ✅ input_data_ref.yml")
print("\n✅ Data Refs 저장 완료")

   ✅ input_data_ref.yml

✅ Data Refs 저장 완료


### 6.7 Run Manifest YAML 생성

In [25]:
# config 파일 해시 계산
config_hashes = {
    'env_yml': get_file_hash(CONF_DIR / 'env.yml'),
    'meta_yml': get_file_hash(CONF_DIR / 'meta.yml'),
    'model_yml': get_file_hash(CONF_DIR / 'model.yml')
}

# S3 경로 생성
ENV = env_config['env']
USER_ID = meta_config['user_id']
PROJECT = meta_config['project']
EXPERIMENT = meta_config['experiment']
REGION = env_config['region']

CONF_BUCKET = env_config['s3']['conf_bucket']
DATA_BUCKET = env_config['s3']['data_bucket']
MODEL_BUCKET = env_config['s3']['model_bucket']

# run_manifest.yml 생성
run_manifest = {
    'run_id': RUN_ID,
    'run_name': f"{EXPERIMENT}-{ALGORITHM}-{SUFFIX}",
    'created_at': CREATED_AT,
    'completed_at': COMPLETED_AT,
    'status': 'completed',
    
    'context': {
        'env': ENV,
        'user_id': USER_ID,
        'project': PROJECT,
        'experiment': EXPERIMENT,
        'model': meta_config['model'],
        'algorithm': ALGORITHM,
        'task': model_config['algorithm']['task'],
        'version': meta_config['version']
    },
    
    's3_refs': {
        'conf_path': f"s3://{CONF_BUCKET}/{ENV}/{USER_ID}/{PROJECT}/{EXPERIMENT}/",
        'data_path': f"s3://{DATA_BUCKET}/{ENV}/{USER_ID}/{PROJECT}/{meta_config['version']}/",
        'model_path': f"s3://{MODEL_BUCKET}/{ENV}/{USER_ID}/{PROJECT}/{EXPERIMENT}/{RUN_ID}/"
    },
    
    'config_hashes': config_hashes,
    
    'runtime': {
        'platform': 'local',
        'python_version': f"{__import__('sys').version_info.major}.{__import__('sys').version_info.minor}",
        'framework': f"lightgbm=={lgb.__version__}"
    },
    
    'summary': {
        'duration_seconds': int((datetime.fromisoformat(COMPLETED_AT) - datetime.fromisoformat(CREATED_AT)).total_seconds()),
        'train_samples': len(X_train),
        'valid_samples': len(X_valid),
        'feature_count': len(feature_cols),
        'best_iteration': model.best_iteration
    },
    
    'artifacts': {
        'config': ['config/env.yml', 'config/meta.yml', 'config/model.yml'],
        'data_refs': ['data_refs/input_data_ref.yml'],
        'model': [f'artifacts/model/model_{SUFFIX}.pkl', f'artifacts/model/model_{SUFFIX}.txt'],
        'metrics': ['artifacts/metrics/model_metrics.yml'],
        'charts': [
            'artifacts/charts/feature_importance.png',
            'artifacts/charts/roc_curve.png',
            'artifacts/charts/confusion_matrix.png',
            'artifacts/charts/learning_curve.png'
        ],
        'explainability': ['artifacts/explainability/feature_impact_summary.png'],
        'reports': ['reports/classification_report.html']
    }
}

save_yaml(run_manifest, OUTPUT_DIRS['metadata'] / 'run_manifest.yml')
print("   ✅ run_manifest.yml")
print("\n✅ Run Manifest 저장 완료")

   ✅ run_manifest.yml

✅ Run Manifest 저장 완료


### 6.8 HTML Report 생성

In [26]:
# HTML 리포트 생성
html_report = f"""
<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Classification Report - {RUN_ID}</title>
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 40px; background: #f5f5f5; }}
        .container {{ max-width: 1200px; margin: 0 auto; background: white; padding: 40px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }}
        h1 {{ color: #2E75B6; border-bottom: 3px solid #2E75B6; padding-bottom: 10px; }}
        h2 {{ color: #404040; margin-top: 30px; }}
        .metrics-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; margin: 20px 0; }}
        .metric-card {{ background: #f8f9fa; padding: 20px; border-radius: 8px; text-align: center; }}
        .metric-value {{ font-size: 2em; font-weight: bold; color: #2E75B6; }}
        .metric-label {{ color: #666; margin-top: 5px; }}
        table {{ width: 100%; border-collapse: collapse; margin: 20px 0; }}
        th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; }}
        th {{ background: #2E75B6; color: white; }}
        tr:hover {{ background: #f5f5f5; }}
        .info {{ background: #e7f3ff; padding: 15px; border-radius: 4px; margin: 10px 0; }}
        .footer {{ margin-top: 40px; padding-top: 20px; border-top: 1px solid #ddd; color: #666; font-size: 0.9em; }}
    </style>
</head>
<body>
    <div class="container">
        <h1>🚢 Titanic Classification Report</h1>
        
        <div class="info">
            <strong>Run ID:</strong> {RUN_ID}<br>
            <strong>Project:</strong> {PROJECT}<br>
            <strong>Experiment:</strong> {EXPERIMENT}<br>
            <strong>Algorithm:</strong> {ALGORITHM}<br>
            <strong>Created:</strong> {CREATED_AT}
        </div>
        
        <h2>📊 Validation Metrics</h2>
        <div class="metrics-grid">
            <div class="metric-card">
                <div class="metric-value">{valid_metrics['accuracy']:.1%}</div>
                <div class="metric-label">Accuracy</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">{valid_metrics['precision']:.1%}</div>
                <div class="metric-label">Precision</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">{valid_metrics['recall']:.1%}</div>
                <div class="metric-label">Recall</div>
            </div>
            <div class="metric-card">
                <div class="metric-value">{valid_metrics['auc_roc']:.4f}</div>
                <div class="metric-label">AUC-ROC</div>
            </div>
        </div>
        
        <h2>📈 Feature Importance (Top 10)</h2>
        <table>
            <tr><th>Rank</th><th>Feature</th><th>Importance</th></tr>
            {''.join(f"<tr><td>{int(row['rank'])}</td><td>{row['feature']}</td><td>{row['importance']:.4f}</td></tr>" for _, row in feature_importance.head(10).iterrows())}
        </table>
        
        <h2>🔢 Confusion Matrix</h2>
        <table>
            <tr><th></th><th>Predicted: Died</th><th>Predicted: Survived</th></tr>
            <tr><td><strong>Actual: Died</strong></td><td>{tn}</td><td>{fp}</td></tr>
            <tr><td><strong>Actual: Survived</strong></td><td>{fn}</td><td>{tp}</td></tr>
        </table>
        
        <h2>⚙️ Model Configuration</h2>
        <table>
            <tr><th>Parameter</th><th>Value</th></tr>
            <tr><td>Algorithm</td><td>{ALGORITHM}</td></tr>
            <tr><td>Max Depth</td><td>{params['max_depth']}</td></tr>
            <tr><td>Learning Rate</td><td>{params['learning_rate']}</td></tr>
            <tr><td>Num Leaves</td><td>{params['num_leaves']}</td></tr>
            <tr><td>Best Iteration</td><td>{model.best_iteration}</td></tr>
            <tr><td>Total Features</td><td>{len(feature_cols)}</td></tr>
        </table>
        
        <div class="footer">
            Generated by GS Retail ML Pipeline | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
        </div>
    </div>
</body>
</html>
"""

with open(OUTPUT_DIRS['reports'] / 'classification_report.html', 'w', encoding='utf-8') as f:
    f.write(html_report)

print("   ✅ classification_report.html")
print("\n✅ HTML Report 생성 완료")

   ✅ classification_report.html

✅ HTML Report 생성 완료


---

## 7. 최종 요약

In [27]:
# Output 디렉토리 구조 출력
def print_tree(path: Path, prefix: str = ""):
    """디렉토리 트리 출력"""
    items = sorted(path.iterdir(), key=lambda x: (x.is_file(), x.name))
    for i, item in enumerate(items):
        is_last = i == len(items) - 1
        connector = "└── " if is_last else "├── "
        print(f"{prefix}{connector}{item.name}")
        if item.is_dir():
            extension = "    " if is_last else "│   "
            print_tree(item, prefix + extension)

print(f"\n📁 Output 디렉토리 구조: {OUTPUT_ROOT}")
print("=" * 60)
print_tree(OUTPUT_ROOT)


📁 Output 디렉토리 구조: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3
├── artifacts
│   ├── charts
│   │   ├── confusion_matrix.png
│   │   ├── feature_importance.png
│   │   ├── learning_curve.png
│   │   └── roc_curve.png
│   ├── explainability
│   │   └── feature_impact_summary.png
│   ├── metrics
│   │   └── model_metrics.yml
│   └── model
│       ├── model_baseline.pkl
│       └── model_baseline.txt
├── config
│   ├── env.yml
│   ├── meta.yml
│   └── model.yml
├── data_refs
│   └── input_data_ref.yml
├── metadata
│   └── run_manifest.yml
├── reports
│   └── classification_report.html
└── executed_notebook.ipynb


In [28]:
# 파일 목록 및 크기
print("\n📄 생성된 파일 목록:")
print("=" * 60)

total_size = 0
for root, dirs, files in os.walk(OUTPUT_ROOT):
    for file in files:
        filepath = Path(root) / file
        size = filepath.stat().st_size
        total_size += size
        rel_path = filepath.relative_to(OUTPUT_ROOT)
        size_str = f"{size / 1024:.1f} KB" if size >= 1024 else f"{size} B"
        print(f"   {rel_path}: {size_str}")

print(f"\n   Total: {total_size / 1024:.1f} KB")


📄 생성된 파일 목록:
   executed_notebook.ipynb: 69.6 KB
   metadata/run_manifest.yml: 1.6 KB
   config/model.yml: 1.4 KB
   config/env.yml: 756 B
   config/meta.yml: 870 B
   reports/classification_report.html: 4.2 KB
   artifacts/charts/learning_curve.png: 100.3 KB
   artifacts/charts/feature_importance.png: 45.8 KB
   artifacts/charts/roc_curve.png: 62.3 KB
   artifacts/charts/confusion_matrix.png: 43.5 KB
   artifacts/model/model_baseline.pkl: 349.9 KB
   artifacts/model/model_baseline.txt: 348.8 KB
   artifacts/metrics/model_metrics.yml: 1.8 KB
   artifacts/explainability/feature_impact_summary.png: 45.9 KB
   data_refs/input_data_ref.yml: 1.2 KB

   Total: 1077.9 KB


In [29]:
print("\n" + "=" * 60)
print("📋 모델링 완료 요약")
print("=" * 60)

print(f"""
🏷️  실행 정보
    - Run ID:     {RUN_ID}
    - Project:    {PROJECT}
    - Experiment: {EXPERIMENT}
    - Algorithm:  {ALGORITHM}

📊 모델 성능 (Validation)
    - Accuracy:  {valid_metrics['accuracy']:.4f}
    - Precision: {valid_metrics['precision']:.4f}
    - Recall:    {valid_metrics['recall']:.4f}
    - F1 Score:  {valid_metrics['f1_score']:.4f}
    - AUC-ROC:   {valid_metrics['auc_roc']:.4f}

📁 Output 경로
    - Local: {OUTPUT_ROOT}
    - S3:    s3://{MODEL_BUCKET}/{ENV}/{USER_ID}/{PROJECT}/{EXPERIMENT}/{RUN_ID}/

🚀 다음 단계
    1. Output 폴더 내용 확인
    2. S3에 업로드 (upload_artifacts.ipynb)
    3. MLflow 등록 (선택)
""")

print("=" * 60)
print(f"⏰ 완료 시각: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


📋 모델링 완료 요약

🏷️  실행 정보
    - Run ID:     20260320_lightgbm_baseline_a9aa9fd3
    - Project:    titanic-survival-prediction
    - Experiment: baseline-awesome-sean-v1
    - Algorithm:  lightgbm

📊 모델 성능 (Validation)
    - Accuracy:  0.8324
    - Precision: 0.8305
    - Recall:    0.7101
    - F1 Score:  0.7656
    - AUC-ROC:   0.8652

📁 Output 경로
    - Local: /home/ec2-user/SageMaker/gs-ds-env/samples/awesome_sean/modeling/output/20260320_lightgbm_baseline_a9aa9fd3
    - S3:    s3://gs-retail-awesome-model-us-west-2/dev/sample/titanic-survival-prediction/baseline-awesome-sean-v1/20260320_lightgbm_baseline_a9aa9fd3/

🚀 다음 단계
    1. Output 폴더 내용 확인
    2. S3에 업로드 (upload_artifacts.ipynb)
    3. MLflow 등록 (선택)

⏰ 완료 시각: 2026-03-20 01:33:27
